# Phase 7: Cuisine Analysis
## Deep Dive into Cuisine Types and Preferences
**Objective:** Analyze cuisine distribution, popularity, and cuisine-quality relationships
**Output:** Cuisine-based insights for menu and business strategy

In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import Counter

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ Libraries imported")

✓ Libraries imported


In [2]:
# Load cleaned data
df = pd.read_csv('Zomato_datasets/zomato_cleaned.csv')

# Extract primary cuisine
df['primary_cuisine'] = df['cuisines'].apply(
    lambda x: str(x).split(',')[0].strip() if pd.notna(x) else 'Unknown'
)

# Count cuisines per restaurant
df['cuisine_count'] = df['cuisines'].apply(
    lambda x: len(str(x).split(',')) if pd.notna(x) else 0
)

print(f"Dataset loaded: {df.shape}")
print(f"Unique primary cuisines: {df['primary_cuisine'].nunique()}")
print(f"\nTop 10 Primary Cuisines:")
print(df['primary_cuisine'].value_counts().head(10))

Dataset loaded: (7105, 14)
Unique primary cuisines: 85

Top 10 Primary Cuisines:
primary_cuisine
North Indian    1943
South Indian     841
Chinese          455
Cafe             449
Biryani          408
Fast Food        395
Continental      253
Bakery           235
Desserts         209
Andhra           193
Name: count, dtype: int64


In [3]:
# 1. Top Cuisines Distribution
top_cuisines = df['primary_cuisine'].value_counts().head(15)

fig = px.bar(
    x=top_cuisines.index,
    y=top_cuisines.values,
    title='Top 15 Cuisines by Restaurant Count',
    labels={'x': 'Cuisine Type', 'y': 'Number of Restaurants'},
    color=top_cuisines.values,
    color_continuous_scale='Sunset'
)

fig.update_layout(height=600, xaxis_tickangle=-45, showlegend=False)
fig.show()

print("\nTop 15 Cuisines:")
print(top_cuisines)
print(f"\nTop 15 cuisines represent {top_cuisines.sum() / len(df) * 100:.1f}% of all restaurants")


Top 15 Cuisines:
primary_cuisine
North Indian    1943
South Indian     841
Chinese          455
Cafe             449
Biryani          408
Fast Food        395
Continental      253
Bakery           235
Desserts         209
Andhra           193
Beverages        149
Kerala           135
Street Food      134
Finger Food       89
Mithai            87
Name: count, dtype: int64

Top 15 cuisines represent 84.1% of all restaurants


In [4]:
# 2. Cuisine Diversity Analysis
cuisine_diversity = df['cuisine_count'].value_counts().sort_index()

fig = px.bar(
    x=cuisine_diversity.index,
    y=cuisine_diversity.values,
    title='Restaurant Distribution by Number of Cuisines Offered',
    labels={'x': 'Number of Cuisines', 'y': 'Count'},
    color=cuisine_diversity.values,
    color_continuous_scale='Blues'
)

fig.update_layout(height=500, showlegend=False)
fig.show()

print(f"\nCuisine Diversity Statistics:")
print(f"  Average cuisines per restaurant: {df['cuisine_count'].mean():.2f}")
print(f"  Median cuisines per restaurant: {df['cuisine_count'].median():.0f}")
print(f"  Max cuisines offered: {df['cuisine_count'].max():.0f}")


Cuisine Diversity Statistics:
  Average cuisines per restaurant: 2.40
  Median cuisines per restaurant: 2
  Max cuisines offered: 8


In [5]:
# 3. Cuisine Performance - Rating Analysis
top_10_cuisines = df['primary_cuisine'].value_counts().head(10).index
df_top = df[df['primary_cuisine'].isin(top_10_cuisines)]

cuisine_ratings = df_top.groupby('primary_cuisine').agg({
    'rating': ['mean', 'std', 'count'],
    'avg_cost': 'mean'
}).reset_index()

cuisine_ratings.columns = ['cuisine', 'avg_rating', 'rating_std', 'restaurant_count', 'avg_cost']
cuisine_ratings = cuisine_ratings.sort_values('avg_rating', ascending=False)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=cuisine_ratings['cuisine'],
    y=cuisine_ratings['avg_rating'],
    marker_color=cuisine_ratings['avg_rating'],
    marker_colorscale='RdYlGn',
    marker_cmin=3,
    marker_cmax=4,
    text=cuisine_ratings['avg_rating'].round(2),
    textposition='auto',
    name='Avg Rating'
))

fig.update_layout(
    title='Average Rating by Top 10 Cuisines',
    xaxis_title='Cuisine Type',
    yaxis_title='Average Rating',
    height=600,
    showlegend=False
)

fig.update_xaxes(tickangle=-45)
fig.show()

print("\nTop 10 Cuisines - Performance Analysis:")
print(cuisine_ratings[['cuisine', 'avg_rating', 'restaurant_count', 'avg_cost']].to_string())


Top 10 Cuisines - Performance Analysis:
        cuisine  avg_rating  restaurant_count     avg_cost
5   Continental    3.813043               253  1114.624506
3          Cafe    3.709577               449   593.986637
6      Desserts    3.607177               209   366.746411
4       Chinese    3.513626               455   549.560440
1        Bakery    3.495745               235   367.106383
9  South Indian    3.449822               841   325.147444
8  North Indian    3.437262              1943   573.443129
0        Andhra    3.436269               193   527.564767
7     Fast Food    3.383797               395   302.911392
2       Biryani    3.345833               408   407.622549


In [6]:
# 4. Price vs Cuisine Analysis
fig = px.box(
    df_top,
    x='primary_cuisine',
    y='avg_cost',
    color='primary_cuisine',
    title='Price Distribution by Top 10 Cuisines',
    labels={'primary_cuisine': 'Cuisine Type', 'avg_cost': 'Average Cost (₹)'},
    points='outliers'
)

fig.update_layout(height=600, xaxis_tickangle=-45, showlegend=False)
fig.show()

In [7]:
# 5. Cuisine Specialization vs Diversification
fig = px.scatter(
    df,
    x='cuisine_count',
    y='rating',
    color='primary_cuisine',
    size='num_ratings',
    hover_name='restaurant_name',
    title='Restaurant Specialization vs Rating (bubble size = num_ratings)',
    labels={'cuisine_count': 'Number of Cuisines', 'rating': 'Rating'},
    color_discrete_sequence=px.colors.qualitative.Set1
)

fig.update_layout(height=600)
fig.show()

# Analyze correlation
corr = df['cuisine_count'].corr(df['rating'])
print(f"\nCorrelation between Cuisine Count and Rating: {corr:.3f}")


Correlation between Cuisine Count and Rating: 0.156


In [8]:
# 6. Cuisine Combinations Analysis
# Extract all individual cuisines
all_cuisines = []
for cuisines_str in df['cuisines']:
    if pd.notna(cuisines_str):
        cuisines = [c.strip() for c in str(cuisines_str).split(',')]
        all_cuisines.extend(cuisines)

cuisine_counter = Counter(all_cuisines)
top_20_individual = dict(cuisine_counter.most_common(20))

fig = px.bar(
    x=list(top_20_individual.keys()),
    y=list(top_20_individual.values()),
    title='Top 20 Individual Cuisines (Including Combinations)',
    labels={'x': 'Cuisine', 'y': 'Frequency'},
    color=list(top_20_individual.values()),
    color_continuous_scale='Viridis'
)

fig.update_layout(height=600, xaxis_tickangle=-45, showlegend=False)
fig.show()

print("\nTop 20 Individual Cuisines:")
for cuisine, count in list(cuisine_counter.most_common(20)):
    print(f"  {cuisine}: {count}")


Top 20 Individual Cuisines:
  North Indian: 3237
  Chinese: 2438
  South Indian: 1464
  Fast Food: 1037
  Biryani: 929
  Continental: 738
  Desserts: 543
  Beverages: 533
  Cafe: 519
  Street Food: 394
  Italian: 383
  Bakery: 304
  Andhra: 292
  Seafood: 266
  Mughlai: 234
  Pizza: 221
  Kerala: 220
  Rolls: 190
  Burger: 189
  Asian: 178


In [9]:
# 7. Restaurant Type by Primary Cuisine
top_cuisines_list = df['primary_cuisine'].value_counts().head(8).index
cuisine_type = df[df['primary_cuisine'].isin(top_cuisines_list)].groupby(['primary_cuisine', 'restaurant_type']).size().reset_index(name='count')
cuisine_type = cuisine_type.sort_values('count', ascending=False).groupby('primary_cuisine').head(4)

fig = px.bar(
    cuisine_type,
    x='count',
    y='primary_cuisine',
    color='restaurant_type',
    title='Top Restaurant Types by Top 8 Cuisines',
    labels={'count': 'Count', 'primary_cuisine': 'Cuisine', 'restaurant_type': 'Restaurant Type'},
    barmode='group'
)

fig.update_layout(height=600)
fig.show()

In [10]:
# 8. Cuisine Market Analysis Summary
print("\n" + "="*70)
print("CUISINE ANALYSIS SUMMARY")
print("="*70)

print(f"\nTotal Unique Primary Cuisines: {df['primary_cuisine'].nunique()}")
print(f"Total Unique Cuisines (including combinations): {len(set(all_cuisines))}")

print(f"\nCuisine Diversity:")
print(f"  Average cuisines per restaurant: {df['cuisine_count'].mean():.2f}")
print(f"  Restaurants with single cuisine: {(df['cuisine_count'] == 1).sum()}")
print(f"  Restaurants with multiple cuisines: {(df['cuisine_count'] > 1).sum()}")

print(f"\n\nTop 5 Cuisines Market Share:")
top_5 = df['primary_cuisine'].value_counts().head(5)
for i, (cuisine, count) in enumerate(top_5.items(), 1):
    pct = count / len(df) * 100
    print(f"  {i}. {cuisine}: {count} restaurants ({pct:.1f}%)")

print("\n✓ Cuisine Analysis Complete!")


CUISINE ANALYSIS SUMMARY

Total Unique Primary Cuisines: 85
Total Unique Cuisines (including combinations): 106

Cuisine Diversity:
  Average cuisines per restaurant: 2.40
  Restaurants with single cuisine: 1823
  Restaurants with multiple cuisines: 5282


Top 5 Cuisines Market Share:
  1. North Indian: 1943 restaurants (27.3%)
  2. South Indian: 841 restaurants (11.8%)
  3. Chinese: 455 restaurants (6.4%)
  4. Cafe: 449 restaurants (6.3%)
  5. Biryani: 408 restaurants (5.7%)

✓ Cuisine Analysis Complete!
